In [ ]:
# SINGLE CELL: PSI/PSR annotation pipeline (NO grouping, NO local/global context)
# - Reads from: messages.json
# - Works on a COPY: <RUN_BASENAME>.json (created once if missing)
# - Writes to disk AFTER EVERY MESSAGE (max resumability)
# - Prints an update AFTER EVERY MESSAGE
# - Exports: <RUN_BASENAME>.xlsx
# - Committee: 4.1-mini/4.1-nano/4.1
# - Escalate/adjudicate: gpt-5.1
# - Tracks: resumable time, per-model + total tokens, and cost (if you fill pricing map)

import os, json, time, re, sys, shutil, uuid, math
from datetime import datetime
from typing import Dict, Any, List, Tuple

# -----------------------------
# 0) Install/import deps
# -----------------------------
def _pip_install(pkgs: List[str]):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

try:
    import pandas as pd
except Exception:
    _pip_install(["pandas", "openpyxl"])
    import pandas as pd

try:
    from dotenv import load_dotenv
except Exception:
    _pip_install(["python-dotenv"])
    from dotenv import load_dotenv

try:
    import tiktoken
except Exception:
    _pip_install(["tiktoken"])
    import tiktoken

try:
    from openai import OpenAI
except Exception:
    _pip_install(["openai"])
    from openai import OpenAI

# -----------------------------
# 1) Config
# -----------------------------
ROOT_DIR = os.getcwd()

SOURCE_MESSAGES_PATH = os.path.join(ROOT_DIR, "messages.json")
RUN_BASENAME         = "25_EEStyle"
WORK_MESSAGES_PATH   = os.path.join(ROOT_DIR, f"{RUN_BASENAME}.json")
XLSX_OUT             = os.path.join(ROOT_DIR, f"{RUN_BASENAME}.xlsx")
STATS_PATH           = os.path.join(ROOT_DIR, f"{RUN_BASENAME}_run_stats.json")

# Committee models (3 different annotators)
COMMITTEE_MODELS = ["gpt-4.1-mini", "gpt-4.1-nano", "gpt-4.1"]  # MUST be odd length
ADJUDICATOR_MODEL = "gpt-5.1"

COMMITTEE_TEMPERATURE   = 0.5    # may be unsupported by some models; we auto-drop if so
ADJUDICATOR_TEMPERATURE = 0.0    # may be unsupported; we auto-drop if so

# When to adjudicate:
# - "any_disagreement": adjudicate whenever committee is not unanimous on ANY label
# - "dimension_or_many": adjudicate if Dimension split OR >=2 other fields split
ADJUDICATE_POLICY = "dimension_or_many"
STORE_COMMITTEE_AUDIT = True

MAX_OUTPUT_TOKENS = 500          # keep tight; schema makes output short anyway

# Fields
BINARY_FIELDS = [
    "DirectAddress", "Attachment", "SelfDisclose", "ReplySeeking",
    "CommRitual", "Monetary", "Backseat", "Emotes"
]
OTHER_FIELDS = ["Dimension"]
ALL_FIELDS = BINARY_FIELDS + OTHER_FIELDS + ["Notes"]

COMMITTEE_SIZE = len(COMMITTEE_MODELS)
if COMMITTEE_SIZE % 2 == 0:
    raise ValueError("COMMITTEE_MODELS must have an odd length for majority voting.")

# -----------------------------
# 1b) Pricing (optional)
# Fill these with YOUR rates (or public rates). If left None, cost=0 but tokens still tracked.
# -----------------------------
MODEL_PRICING_PER_1M = {
    "gpt-5.1":    {"input": 1.25, "cached": 0.125, "output": 10.00},
    "gpt-4.1-mini": {"input": 0.4, "cached": 0.1, "output": 1.60},
    "gpt-4.1-nano": {"input": 0.1, "cached": 0.025, "output": 0.40},
    "gpt-4.1":    {"input": 2.00, "cached": 0.50,  "output": 8.00},
}

DEFAULT_MODEL_PRICING = {"input": None, "cached": None, "output": None}

# -----------------------------
# 2) Prompt
# -----------------------------
# -----------------------------
# 2) Prompt
# -----------------------------
DEVELOPER_PROMPT = r"""
You are an annotation model. Your task is to label each Twitch chat message with a set of binary/ternary fields using the provided schema.
Follow definitions and decision rules exactly. Return ONLY a single JSON object (no wrapper object, no array, no text).

OUTPUT FORMAT (STRICT)
Return ONLY ONE JSON object with keys:
DirectAddress, Attachment, SelfDisclose, ReplySeeking, CommRitual, Monetary, Backseat, Emotes, Dimension, Notes.

Each label field must be one of:
"Y" = present
""  = not present / leave blank

For Dimension, use exactly one of:
"P" if positive/affiliative stance toward streamer/community
"N" if negative/hostile stance toward streamer/community
No neutral/mixed category. If mixed, label based on the dominant intent (“punchline”).

WHAT YOU ARE ANNOTATING
Annotate message-level expressed cues in livestream chat from Twitch. Multi-label is allowed.
Do not infer hidden intent beyond the text and immediate conversational form.

CATEGORY DECISION RULES
A) Dimension (valence)
Set Dimension independently from other categories.
P (Positive): praise/support, gratitude, affection, encouragement, admiration; playful teasing that is clearly supportive.
N (Negative): insults, hostility, shaming, threats, aggressive blame; hate-watch enjoyment of failure.
Mixed: choose dominant intent.

B) PSI/PSR-leaning categories (multi-label)
1) Direct Address / second-person to streamer -> DirectAddress
Mark "Y" only when the message targets the streamer as an addressee:
- explicit @streamer
- streamer name used as addressee (“Emiru, …”)
- second-person pronouns clearly referring to streamer not another person (“you/your/u”) with directed statement/command
Avoid false positives:
- talking about streamer: “emiru is cracked”
- addressing chat: “you guys…”
- ambiguous “you” -> blank unless strong cue (name/@mention).

Important Point! You should keep an eye on morphs, euphemisms, and varying forms of chatting most common for Gen-Z, like "ya", "yo" etc.

2) Attachment / affection / relational warmth -> Attachment
Mark "Y" for warmth/care/pride/missing/love toward streamer or relational closeness markers.
Attachment-strengtheners (no extra labels; just annotate Attachment when applicable):
- longitudinal persistence beyond exposure: off-stream anticipation, absence felt, “all week,” “can’t wait,” “glad you’re back”
- investment/loyalty talk: “watch every stream,” “since 2018,” “never miss a VOD,” schedule organized around streams
- social realism / real-life tie framing: “you’re like family,” “real friend,” “like a brother/sister”
- anthropomorphism/personification: “you care about us,” “she understands me,” “he worries about chat”
Avoid false positives:
- “love this game” (not streamer-directed)
- generic hype alone (“Pog”, “Lets go”) unless clearly relational.

3) Identification / self-disclosure / homophily -> SelfDisclose
Mark "Y" when viewer shares personal info or similarity claims aimed at relational connection:
- “I’m also…”, “same here…”, “as a fellow…”, "I also do that..."
- personal disclosure positioned as sharing-with-streamer (“I had a rough day, thanks for being here”)
- “I have an exam tomorrow, wish me luck” can count if framed as a bid for connection (even without explicit “you”).
Strengthener (no new label):
- knowledge/narrative recall is not SelfDisclose by itself; use it to support DirectAddress/Attachment/ReplySeeking if it functions that way.

4) Reciprocity bids / reply-seeking -> ReplySeeking
Mark "Y" when seeking acknowledgment/response/recognition:
- “notice me,” “read this,” “answer please,” “remember me,” “did you see my last message,” “thanks for replying last week,” "why you not seeing me"
- Or asking them to do something ordinary like drinking water or blink for safety of eyes. 
Avoid false positives:
- generic info questions not aimed at streamer (“what game is this?”) unless streamer-directed by name/@mention . 

5) Community ritual talk & co-creation -> CommRitual
Mark "Y" for shared rituals/inside jokes/coordinated norms tied to community:
- “spam LUL,” “copy pasta time,” “chat do the thing,” “only real ones remember…,” “as always…,” "Let's go..."
Avoid false positives:
- emotes alone ≠ ritual talk without ritual framing.

6) Monetary support actions -> Monetary
Mark "Y" for subs/donos/gifts/streaks or accompanying text:
- “subbed X months,” “gifted subs,” “dono,” “renewed,” “superchat”
Hard rule: If the message tag is "USERNOTICE" instead of "PRIVATEMESSAGE" ALWAYS mark Monetary="Y".
Avoid false positives:
- unrelated money talk (“costs $60”).

ADDED CATEGORIES (behavioral / surface form)
7) Backseat guidance -> Backseat
Mark "Y" for gameplay/stream/content direction/coaching/strategy/idea:
- “go left,” “use ult,” “buy armor,” “skip cutscene,” “watch out…,” "You have to change it in the menu"
Or general questions asking regarding the topic:
- "Hassan are they capitalists?", "you should do a comparison on their technical data"
Backseat can co-occur with DirectAddress, but backseating is not parasocial by itself.

8) Emotes/emoji presence -> Emotes
Mark "Y" if any Twitch-style emote tokens or unicode emojis appear, including <3, and tokens like Kappa, pogchamp, KEKW, etc.

CONTRASTIVE DEMONSTRATIONS (BEHAVIORAL TARGETS)
Follow the demos for common pitfalls:
Demo A — DirectAddress vs about-talk; “you” ambiguity
General: emiru is the streamer; “chat” = audience. Local: streamer just returned; greetings happening.
Inputs:
(1) "Everyone missed you beautiful Emiru <3 <3 WutFace"
(2) "emiru is cracked today"
(3) "you guys missed her too right? lol"
Correct:
(1) DirectAddress=Y (name + “you”), Attachment=Y (“missed you”, warmth), Emotes=Y, Dimension=P
(2) About-talk only: DirectAddress blank; Dimension=P (praise)
(3) Targets chat (“you guys”): DirectAddress blank; Dimension=P (light/affiliative)
Avoid:
- Don’t set DirectAddress just because streamer name appears (2)
- Don’t set DirectAddress from “you” when it clearly addresses chat (3)

Demo B — Loyalty/investment as Attachment; recall ≠ SelfDisclose; mixed valence punchline rule
General: hasanabi political streams; “since YEAR” indicates tenure.
Inputs:
(10) "Been watching since 2019. Never miss a VOD."
(11) "Remember that 2019 meltdown? downhill ever since lol"
(12) "I love you but you’re being an idiot tonight"
Correct:
(10) Attachment=Y (investment/loyalty), SelfDisclose=Y, Dimension=P
(11) Dimension=N (criticism); SelfDisclose blank (recall is not personal disclosure)
(12) Attachment=Y (“love you”), Dimension=N (dominant thrust is insult/punchline)
Avoid:
- Don’t force SelfDisclose from narrative recall (11)
- Don’t label Dimension=P just because affection phrase exists (12)

Demo C — USERNOTICE monetary override + co-occurrence
Rule: msgTag="USERNOTICE" => Monetary=Y ALWAYS.
Input:
(20) msgTag=USERNOTICE, "Renewed my sub for 24 months — love you Emiru <3"
Correct:
Monetary=Y (forced), Attachment=Y, DirectAddress=Y (name used as addressee), Emotes=Y (<3), Dimension=P
Avoid:
- Never leave Monetary blank on USERNOTICE even if message is mostly affection

Return ONLY a JSON array, no text, no code fences.
"""

# -----------------------------
# 3) Structured output schema (THIS is the “schema” that actually matters)
#    Matches ONE message -> ONE object
# -----------------------------
LABEL_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "DirectAddress": {"type": "string", "enum": ["Y", ""]},
        "Attachment":    {"type": "string", "enum": ["Y", ""]},
        "SelfDisclose":  {"type": "string", "enum": ["Y", ""]},
        "ReplySeeking":  {"type": "string", "enum": ["Y", ""]},
        "CommRitual":    {"type": "string", "enum": ["Y", ""]},
        "Monetary":      {"type": "string", "enum": ["Y", ""]},
        "Backseat":      {"type": "string", "enum": ["Y", ""]},
        "Emotes":        {"type": "string", "enum": ["Y", ""]},
        "Dimension":     {"type": "string", "enum": ["P", "N"]},
        "Notes":         {"type": "string"},
    },
    "required": ALL_FIELDS
}

# Try to enforce structured outputs via Responses API.
# If your SDK/model rejects this, we auto-fallback (see call_with_retries()).
STRUCTURED_TEXT_FORMAT = {
    "type": "json_schema",
    "name": "psi_psr_labels",
    "strict": True,
    "schema": LABEL_SCHEMA
}
USE_STRUCTURED_OUTPUTS = True

# -----------------------------
# 4) Env + client
# -----------------------------
load_dotenv(os.path.join(ROOT_DIR, ".env"))
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found. Ensure .env exists with OPENAI_API_KEY=...")

client = OpenAI()

# -----------------------------
# 5) Helpers
# -----------------------------
def now_ts():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def safe_write_json(path: str, obj: Any, retries: int = 10, sleep: float = 0.05):
    directory = os.path.dirname(path)
    os.makedirs(directory, exist_ok=True)
    tmp = f"{path}.{uuid.uuid4().hex}.tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())
    for _ in range(retries):
        try:
            shutil.move(tmp, path)
            return
        except PermissionError:
            time.sleep(sleep)
    raise PermissionError(f"Could not replace {path} after {retries} retries")

def load_json_if_exists(path: str, default):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default

def ensure_working_copy():
    if not os.path.exists(SOURCE_MESSAGES_PATH):
        raise FileNotFoundError(f"messages.json not found at: {SOURCE_MESSAGES_PATH}")
    if not os.path.exists(WORK_MESSAGES_PATH):
        shutil.copy2(SOURCE_MESSAGES_PATH, WORK_MESSAGES_PATH)
        print(f"[{now_ts()}] Created working copy: {WORK_MESSAGES_PATH}")
    else:
        print(f"[{now_ts()}] Using existing working copy (resume): {WORK_MESSAGES_PATH}")

def load_messages() -> List[Dict[str, Any]]:
    with open(WORK_MESSAGES_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Working messages file must be a JSON array of message objects.")
    return data

def is_annotated(msg: Dict[str, Any]) -> bool:
    return str(msg.get("Annotated", "")).strip().lower() == "true"

def mark_annotated(msg: Dict[str, Any]):
    msg["Annotated"] = "true"

def ensure_fields(msg: Dict[str, Any]):
    for k in BINARY_FIELDS + OTHER_FIELDS + ["Notes"]:
        msg.setdefault(k, "")
    for k in ["raw_msg", "msgTag", "streamer", "Annotated"]:
        msg.setdefault(k, "")
    if msg["Annotated"] == "":
        msg["Annotated"] = "False"

def enforce_binary(v: Any) -> str:
    return "Y" if str(v).strip().upper() == "Y" else ""

def enforce_dimension(v: Any, fallback: str = "P") -> str:
    s = str(v).strip().upper()
    if s in ("P", "POS", "POSITIVE"):
        return "P"
    if s in ("N", "NEG", "NEGATIVE"):
        return "N"
    return fallback

def clip_notes(text: str, max_words: int = 20) -> str:
    t = re.sub(r"\s+", " ", (text or "").strip())
    if not t:
        return ""
    words = t.split(" ")
    return t if len(words) <= max_words else (" ".join(words[:max_words]).rstrip() + "…")

def short_preview(s: str, n: int = 110) -> str:
    s = re.sub(r"\s+", " ", (s or "").strip())
    return s if len(s) <= n else (s[:n-1] + "…")

# --- token encoding (only for local utility; API usage comes from response.usage)
def get_encoding():
    try:
        return tiktoken.encoding_for_model("gpt-4.1")
    except Exception:
        return tiktoken.get_encoding("o200k_base")
ENC = get_encoding()

def usage_breakdown(resp) -> Dict[str, Any]:
    u = getattr(resp, "usage", None)
    if not u:
        return {}
    try:
        return u if isinstance(u, dict) else u.model_dump()
    except Exception:
        try:
            return dict(u)
        except Exception:
            return {"raw_usage": str(u)}

def usage_triplet(resp) -> Dict[str, int]:
    ud = usage_breakdown(resp)
    in_tok  = int(ud.get("input_tokens") or 0)
    out_tok = int(ud.get("output_tokens") or 0)
    cached_tok = 0
    try:
        cached_tok = int((ud.get("input_tokens_details") or {}).get("cached_tokens") or 0)
    except Exception:
        cached_tok = 0
    return {"in": in_tok, "out": out_tok, "cached": cached_tok}

def _ensure_model_buckets(stats: Dict[str, Any]):
    stats.setdefault("per_model_tokens", {})
    if not isinstance(stats["per_model_tokens"], dict):
        stats["per_model_tokens"] = {}
    # pre-seed known models (and we also support new ones dynamically)
    for m in set(list(MODEL_PRICING_PER_1M.keys()) + COMMITTEE_MODELS + [ADJUDICATOR_MODEL]):
        stats["per_model_tokens"].setdefault(m, {"input_tokens": 0, "output_tokens": 0, "cached_tokens": 0})

def accumulate_usage_by_model(stats: Dict[str, Any], resp, model_name: str) -> Dict[str, int]:
    u = usage_triplet(resp)
    stats["total_input_tokens"]  += u["in"]
    stats["total_output_tokens"] += u["out"]
    stats["total_cached_tokens"] += u["cached"]
    stats.setdefault("per_model_tokens", {})
    stats["per_model_tokens"].setdefault(model_name, {"input_tokens": 0, "output_tokens": 0, "cached_tokens": 0})
    stats["per_model_tokens"][model_name]["input_tokens"]  += u["in"]
    stats["per_model_tokens"][model_name]["output_tokens"] += u["out"]
    stats["per_model_tokens"][model_name]["cached_tokens"] += u["cached"]
    return u

def estimate_cost_usd_per_model(input_tok: int, cached_tok: int, output_tok: int, pricing: Dict[str, float]) -> float:
    inp = max(0, int(input_tok or 0))
    cached = max(0, int(cached_tok or 0))
    out = max(0, int(output_tok or 0))
    cached = min(cached, inp)
    non_cached = inp - cached
    pin  = pricing.get("input", 0.0)  or 0.0
    pc   = pricing.get("cached", 0.0) or 0.0
    pout = pricing.get("output", 0.0) or 0.0
    return (non_cached/1e6)*pin + (cached/1e6)*pc + (out/1e6)*pout

def estimate_total_cost_from_stats(stats: Dict[str, Any]) -> float:
    total = 0.0
    for model_name, toks in (stats.get("per_model_tokens", {}) or {}).items():
        pricing = MODEL_PRICING_PER_1M.get(model_name, DEFAULT_MODEL_PRICING)
        total += estimate_cost_usd_per_model(
            toks.get("input_tokens", 0),
            toks.get("cached_tokens", 0),
            toks.get("output_tokens", 0),
            pricing
        )
    return total

def format_per_model_cost_breakdown(stats: Dict[str, Any]) -> str:
    lines = []
    per_model = stats.get("per_model_tokens", {}) or {}
    for model_name in sorted(per_model.keys()):
        toks = per_model[model_name]
        pricing = MODEL_PRICING_PER_1M.get(model_name, DEFAULT_MODEL_PRICING)
        c = estimate_cost_usd_per_model(
            toks.get("input_tokens", 0),
            toks.get("cached_tokens", 0),
            toks.get("output_tokens", 0),
            pricing
        )
        lines.append(
            f"    {model_name}: in={toks.get('input_tokens',0)} cached={toks.get('cached_tokens',0)} "
            f"out={toks.get('output_tokens',0)} | est_cost=${c:.6f}"
        )
    return "\n".join(lines)

def load_stats() -> Dict[str, Any]:
    stats = load_json_if_exists(STATS_PATH, default={})
    if not isinstance(stats, dict):
        stats = {}
    stats.setdefault("accumulated_seconds", 0.0)
    stats.setdefault("runs", 0)
    stats.setdefault("total_input_tokens", 0)
    stats.setdefault("total_output_tokens", 0)
    stats.setdefault("total_cached_tokens", 0)
    stats.setdefault("last_run_started_at", "")
    stats.setdefault("last_run_ended_at", "")
    _ensure_model_buckets(stats)
    return stats

def save_stats(stats: Dict[str, Any]):
    safe_write_json(STATS_PATH, stats)

# ---- JSON parsing (kept robust even if schema is enforced)
def try_repair_json(text: str) -> str:
    t = (text or "").strip()
    t = re.sub(r"^```json\s*", "", t, flags=re.IGNORECASE)
    t = re.sub(r"^```\s*", "", t)
    t = re.sub(r"\s*```$", "", t)
    t = t.strip()
    m_obj = re.search(r"\{.*\}", t, flags=re.DOTALL)
    if m_obj:
        t = m_obj.group(0)
    t = t.replace("\t", " ")
    t = re.sub(r",\s*([}\]])", r"\1", t)
    return t.strip()

def get_output_text(resp) -> str:
    if hasattr(resp, "output_text") and resp.output_text:
        return resp.output_text
    out = getattr(resp, "output", None) or []
    for item in out:
        if isinstance(item, dict) and item.get("type") == "message":
            for c in (item.get("content") or []):
                if isinstance(c, dict) and c.get("type") in ("output_text", "text"):
                    return c.get("text", "") or ""
    return ""

def reformat_to_json_only(model_name: str, bad_text: str) -> str:
    fix_prompt = (
        "Rewrite the following into VALID JSON ONLY.\n"
        "Do not change meanings or labels. Do not omit keys.\n"
        "Output ONLY valid JSON.\n\nTEXT:\n" + bad_text
    )
    kwargs = {
        "model": model_name,
        "input": [{"role": "user", "content": fix_prompt}],
        "max_output_tokens": 400,
    }
    resp = call_with_retries(kwargs)
    accumulate_usage_by_model(stats, resp, model_name)
    return (get_output_text(resp) or "").strip()

def parse_single_json(model_name: str, resp) -> Dict[str, Any]:
    text = get_output_text(resp)
    if not text:
        raise RuntimeError("Empty model output; cannot parse JSON.")
    t = try_repair_json(text)
    try:
        obj = json.loads(t)
    except Exception:
        fixed = reformat_to_json_only(model_name, text)
        fixed2 = try_repair_json(fixed)
        obj = json.loads(fixed2)

    if not isinstance(obj, dict):
        raise RuntimeError(f"Expected JSON object, got {type(obj)}. text={text[:300]}")

    # Normalize and enforce
    for f in BINARY_FIELDS:
        obj[f] = enforce_binary(obj.get(f, ""))
    obj["Dimension"] = enforce_dimension(obj.get("Dimension", ""), fallback="P")
    obj["Notes"] = clip_notes(obj.get("Notes", ""), max_words=20)

    # Ensure all keys exist
    for k in ALL_FIELDS:
        obj.setdefault(k, "")

    return obj

# ---- Committee math
def normalized_entropy(counts: Dict[str, int]) -> float:
    total = sum(counts.values())
    if total <= 0:
        return 0.0
    ps = [v/total for v in counts.values() if v > 0]
    if len(ps) <= 1:
        return 0.0
    H = -sum(p * math.log(p) for p in ps)
    return float(H / math.log(2.0))

def vote_committee(outputs: List[Dict[str, Any]]) -> Tuple[Dict[str, str], Dict[str, Dict[str, int]], Dict[str, float]]:
    m = len(outputs)
    voted, counts, ent = {}, {}, {}

    for f in BINARY_FIELDS:
        y = sum(1 for o in outputs if enforce_binary(o.get(f, "")) == "Y")
        c = {"Y": y, "": m - y}
        counts[f] = c
        ent[f] = normalized_entropy(c)
        voted[f] = "Y" if y > (m/2.0) else ""

    p = sum(1 for o in outputs if enforce_dimension(o.get("Dimension",""), "P") == "P")
    n = m - p
    c = {"P": p, "N": n}
    counts["Dimension"] = c
    ent["Dimension"] = normalized_entropy(c)
    voted["Dimension"] = "P" if p > (m/2.0) else "N"
    return voted, counts, ent

def should_adjudicate(vote_counts: Dict[str, Dict[str, int]], m: int) -> bool:
    split = {f: (max(c.values()) < m) for f, c in vote_counts.items()}
    if ADJUDICATE_POLICY == "any_disagreement":
        return any(split.values())
    if split.get("Dimension", False):
        return True
    other_splits = sum(1 for k, v in split.items() if k != "Dimension" and v)
    return other_splits >= 2

# ---- API call builder
def build_kwargs(model_name: str, user_payload: Dict[str, Any], mode: str, temperature: float | None) -> Dict[str, Any]:
    """
    mode: "member" or "adjudicator"
    """
    if mode == "member":
        user_text = (
            "Annotate this SINGLE message.\n"
            "Return ONLY the JSON object with keys:\n"
            + ", ".join(ALL_FIELDS) + ".\n\n"
            "INPUT_MESSAGE:\n" + json.dumps(user_payload, ensure_ascii=False)
        )
    else:
        # adjudicator sees candidates (provided separately by caller)
        raise ValueError("Use build_adjudicator_kwargs for adjudicator.")

    kwargs = dict(
        model=model_name,
        input=[
            {"role": "developer", "content": DEVELOPER_PROMPT},
            {"role": "user", "content": user_text},
        ],
        max_output_tokens=MAX_OUTPUT_TOKENS,
    )

    if temperature is not None:
        kwargs["temperature"] = temperature

    if USE_STRUCTURED_OUTPUTS:
        # Try to enforce schema (auto-dropped if unsupported)
        kwargs["text"] = {"format": STRUCTURED_TEXT_FORMAT}

    return kwargs

def build_adjudicator_kwargs(user_payload: Dict[str, Any], committee_outputs: List[Dict[str, Any]], vote_counts: Dict[str, Dict[str,int]], temperature: float | None) -> Dict[str, Any]:
    adjudicate_prompt = (
        "You are the ADJUDICATOR.\n"
        "Given ONE message and committee candidates, output the BEST final labels under the definitions.\n"
        "You MAY override majority if majority violates rules.\n"
        "Return ONLY the JSON object with keys:\n"
        + ", ".join(ALL_FIELDS) + ".\n\n"
        "INPUT_MESSAGE:\n" + json.dumps(user_payload, ensure_ascii=False) + "\n\n"
        "VOTE_COUNTS:\n" + json.dumps(vote_counts, ensure_ascii=False) + "\n\n"
        "COMMITTEE_CANDIDATES:\n" + json.dumps(committee_outputs, ensure_ascii=False)
    )

    kwargs = dict(
        model=ADJUDICATOR_MODEL,
        input=[
            {"role": "developer", "content": DEVELOPER_PROMPT},
            {"role": "user", "content": adjudicate_prompt},
        ],
        max_output_tokens=MAX_OUTPUT_TOKENS,
    )

    if temperature is not None:
        kwargs["temperature"] = temperature

    if USE_STRUCTURED_OUTPUTS:
        kwargs["text"] = {"format": STRUCTURED_TEXT_FORMAT}

    return kwargs

# ---- Robust retries that also AUTO-DROP unsupported parameters (fixes your temperature crash)
def call_with_retries(kwargs: Dict[str, Any], tries: int = 6, base_sleep: float = 1.5):
    last_err = None
    for attempt in range(1, tries + 1):
        try:
            return client.responses.create(**kwargs)
        except Exception as e:
            last_err = e
            msg = str(e)

            # Auto-drop unsupported parameters (e.g., temperature)
            m = re.search(r"Unsupported parameter:\s*'([^']+)'", msg)
            if m:
                bad_param = m.group(1)
                if bad_param in kwargs:
                    kwargs.pop(bad_param, None)
                    print(f"[{now_ts()}]   WARN: dropped unsupported param '{bad_param}' and retrying immediately.")
                    continue

            # Sometimes the SDK error string formats slightly differently
            m2 = re.search(r"Unsupported parameter:\s*\"([^\"]+)\"", msg)
            if m2:
                bad_param = m2.group(1)
                if bad_param in kwargs:
                    kwargs.pop(bad_param, None)
                    print(f"[{now_ts()}]   WARN: dropped unsupported param \"{bad_param}\" and retrying immediately.")
                    continue

            sleep = base_sleep * (2 ** (attempt - 1))
            print(f"[{now_ts()}]   ERROR (attempt {attempt}/{tries}): {e}. Sleeping {sleep:.1f}s then retry...")
            time.sleep(sleep)

    raise last_err

# -----------------------------
# 6) Setup working copy + load data + stats
# -----------------------------
ensure_working_copy()
messages = load_messages()
for m in messages:
    ensure_fields(m)

total = len(messages)
remaining = sum(1 for m in messages if not is_annotated(m))
print(f"[{now_ts()}] Loaded {total} messages from working copy. Remaining unannotated: {remaining}")

stats = load_stats()
stats["runs"] += 1
stats["last_run_started_at"] = now_ts()
save_stats(stats)

run_start = time.time()
newly_annotated = 0

# -----------------------------
# 7) Main loop
# -----------------------------
for idx, msg in enumerate(messages):
    if is_annotated(msg):
        continue

    raw_msg = msg.get("raw_msg", "") or ""
    msgTag  = msg.get("msgTag", "") or ""

    if not raw_msg.strip() and not msgTag.strip():
        mark_annotated(msg)
        newly_annotated += 1
        safe_write_json(WORK_MESSAGES_PATH, messages)
        elapsed = time.time() - run_start
        stats["accumulated_seconds"] += elapsed
        run_start = time.time()
        stats["last_run_ended_at"] = now_ts()
        save_stats(stats)
        print(f"[{now_ts()}] #{idx} EMPTY -> marked Annotated=true | newly_annotated={newly_annotated}")
        continue

    user_payload = {"msgTag": msgTag, "raw_msg": raw_msg}

    # -----------------------------
    # Committee: 3 different models
    # -----------------------------
    committee_outputs = []
    committee_lat = 0.0
    usage_totals = {"in": 0, "out": 0, "cached": 0}

    for j, model_name in enumerate(COMMITTEE_MODELS, start=1):
        member_kwargs = build_kwargs(
            model_name=model_name,
            user_payload=user_payload,
            mode="member",
            temperature=COMMITTEE_TEMPERATURE
        )

        t0 = time.time()
        resp_m = call_with_retries(member_kwargs)
        committee_lat += (time.time() - t0)

        u_m = accumulate_usage_by_model(stats, resp_m, model_name=model_name)
        usage_totals["in"] += u_m["in"]
        usage_totals["out"] += u_m["out"]
        usage_totals["cached"] += u_m["cached"]

        o_m = parse_single_json(model_name, resp_m)
        if STORE_COMMITTEE_AUDIT:
            o_m["_model"] = model_name
        committee_outputs.append(o_m)

    voted, vote_counts, entropies = vote_committee(committee_outputs)
    do_adj = should_adjudicate(vote_counts, COMMITTEE_SIZE)

    # -----------------------------
    # Adjudication -> gpt-5.1
    # -----------------------------
    adjudicated = False
    final_obj = dict(voted)
    final_obj["Notes"] = committee_outputs[0].get("Notes", "")

    if do_adj:
        adj_kwargs = build_adjudicator_kwargs(
            user_payload=user_payload,
            committee_outputs=committee_outputs,
            vote_counts=vote_counts,
            temperature=ADJUDICATOR_TEMPERATURE
        )
        t0 = time.time()
        resp_a = call_with_retries(adj_kwargs)
        committee_lat += (time.time() - t0)

        u_a = accumulate_usage_by_model(stats, resp_a, model_name=ADJUDICATOR_MODEL)
        usage_totals["in"] += u_a["in"]
        usage_totals["out"] += u_a["out"]
        usage_totals["cached"] += u_a["cached"]

        out_a = parse_single_json(ADJUDICATOR_MODEL, resp_a)
        for f in BINARY_FIELDS:
            final_obj[f] = enforce_binary(out_a.get(f, ""))
        final_obj["Dimension"] = enforce_dimension(out_a.get("Dimension", ""), fallback=voted["Dimension"])
        final_obj["Notes"] = clip_notes(out_a.get("Notes", ""), max_words=20)
        adjudicated = True

    # -----------------------------
    # Apply final labels to msg
    # -----------------------------
    for f in BINARY_FIELDS:
        msg[f] = enforce_binary(final_obj.get(f, ""))

    # Monetary override (hard rule)
    if str(msgTag).strip().upper() == "USERNOTICE":
        msg["Monetary"] = "Y"

    msg["Dimension"] = enforce_dimension(final_obj.get("Dimension", ""), fallback="P")
    msg["Notes"] = clip_notes(final_obj.get("Notes", ""), max_words=20)

    if STORE_COMMITTEE_AUDIT:
        msg["Adjudicated"] = "true" if adjudicated else "false"
        msg["CommitteeVote"] = json.dumps(vote_counts, ensure_ascii=False)
        msg["CommitteeEntropy"] = json.dumps(entropies, ensure_ascii=False)
        # optional: keep candidates (big). Uncomment if you really want it.
        # msg["CommitteeCandidates"] = json.dumps(committee_outputs, ensure_ascii=False)

    # Mark + persist
    mark_annotated(msg)
    newly_annotated += 1
    safe_write_json(WORK_MESSAGES_PATH, messages)

    elapsed_segment = time.time() - run_start
    stats["accumulated_seconds"] += elapsed_segment
    run_start = time.time()
    stats["last_run_ended_at"] = now_ts()
    _ensure_model_buckets(stats)
    save_stats(stats)

    total_done = sum(1 for m in messages if is_annotated(m))
    cost_now = estimate_total_cost_from_stats(stats)

    print(
        f"[{now_ts()}] #{idx} streamer='{(msg.get('streamer') or '').strip()}' tag='{msgTag.strip()}' "
        f"msg='{short_preview(raw_msg, 110)}'\n"
        f"  -> DA={msg['DirectAddress'] or '-'} Att={msg['Attachment'] or '-'} SD={msg['SelfDisclose'] or '-'} "
        f"RS={msg['ReplySeeking'] or '-'} CR={msg['CommRitual'] or '-'} Mon={msg['Monetary'] or '-'} "
        f"Back={msg['Backseat'] or '-'} Emo={msg['Emotes'] or '-'} Dim={msg['Dimension'] or '-'} "
        f"Notes='{msg['Notes']}'\n"
        f"  committee: models={COMMITTEE_MODELS} adjudicated={'Y' if adjudicated else 'N'} | latency_total={committee_lat:.2f}s\n"
        f"  usage (this msg): in={usage_totals['in']} out={usage_totals['out']} cached_in={usage_totals['cached']}\n"
        f"  totals: annotated={total_done}/{total} | newly_annotated_this_run={newly_annotated} | est_cost_total=${cost_now:.6f}\n"
    )

# -----------------------------
# 8) Export XLSX
# -----------------------------
df = pd.DataFrame(messages)
front = ["streamer", "msgTag", "raw_msg", "Annotated"] + BINARY_FIELDS + ["Dimension", "Notes"]
audit = ["Adjudicated", "CommitteeVote", "CommitteeEntropy"]
front2 = [c for c in front + audit if c in df.columns]
cols = front2 + [c for c in df.columns if c not in front2]
df = df[cols]
df.to_excel(XLSX_OUT, index=False)

# -----------------------------
# 9) Final summary
# -----------------------------
remaining = sum(1 for m in messages if not is_annotated(m))
total_cost = estimate_total_cost_from_stats(stats)

print(f"\n[{now_ts()}] DONE.")
print(f"[{now_ts()}] Total messages={total} | Newly annotated this run={newly_annotated} | Remaining={remaining}")
print(f"[{now_ts()}] Working JSON: {WORK_MESSAGES_PATH}")
print(f"[{now_ts()}] XLSX: {XLSX_OUT}")
print(f"[{now_ts()}] Resumable time (accumulated): {stats['accumulated_seconds']:.2f} seconds")
print(f"[{now_ts()}] Tokens total: input={stats['total_input_tokens']}, cached_input={stats['total_cached_tokens']}, output={stats['total_output_tokens']}")
print(f"[{now_ts()}] Estimated total cost (filled pricing only): ${total_cost:.6f}")
print(f"[{now_ts()}] Per-model breakdown:\n{format_per_model_cost_breakdown(stats)}")
print(f"[{now_ts()}] Run stats file: {STATS_PATH}")
